# DS-01 · Galton (1886) — Regression Towards Mediocrity in Hereditary Stature

**Paper:** Francis Galton (1886), *Regression Towards Mediocrity in Hereditary Stature*.  
**Tier:** Intro · **Kerjain di Google Colab, lalu `File → Save a copy in GitHub`.**

---
### 📋 Brief (baca pelan-pelan, ini konteksnya)

Galton ngukur tinggi ribuan orang tua dan anaknya. Dia nemu pola aneh:
orang tua **sangat tinggi** punya anak yang **tinggi juga — tapi biasanya lebih pendek dari orang tuanya**, 
lebih mepet ke rata-rata orang kebanyakan. Yang **sangat pendek** kebalikannya: anaknya pendek, tapi agak lebih tinggi, naik ke arah rata-rata.

Galton nyebut gejala ini *"regression towards mediocrity"* (ketarik balik ke tengah). Dari sinilah kata **"regresi"** di statistika lahir. 
Kalau lu gambar garis lurus terbaik lewat titik-titik (tinggi ortu, tinggi anak), **kemiringan garisnya di bawah 1** — itulah bukti angkanya.

**Tujuan lu di notebook ini:** bikin **garis lurus terbaik (regresi linear / OLS) dari NOL pakai numpy**, 
terus buktiin sendiri kalau kemiringannya kira-kira **0.6** (bukan 1).

> Kenapa dari nol, bukan pakai library jadi? Karena tujuannya *ngerti mesinnya*, bukan mencet tombol. Nanti di paper-paper berikutnya mesin ini kepake terus.


### 🧠 Intuisi dulu, sebelum sentuh kode

Bayangin garis lurus: **`tinggi_anak = α + β × tinggi_ortu`**
- **β (beta)** = *kemiringan* garis. "Tiap 1 inci ortu lebih tinggi, anak lebih tinggi berapa inci?"
- **α (alfa)** = *titik potong* — tinggi anak kalau tinggi ortu = 0 (nggak masuk akal fisiknya, cuma penggeser garis).

Kalau anak = **persis** copy ortu, β = **1** (naik 1 inci, anak naik 1 inci). 
Temuan Galton: β cuma **~0.6**. Artinya ortu 10 inci di atas rata-rata → anak cuma ~6 inci di atas rata-rata. **Ketarik balik ke tengah.** Itu inti seluruh latihan ini.

Semua Task di bawah cuma cara berbeda buat **nyari angka β dan α terbaik** dari data.


### ✅ Rubric — Definition of Done (target biar dianggap lulus)

| # | Yang harus ada | Lulus kalau |
|---|---|---|
| 1 | Load data (sel Setup) | `n` kecetak, `df.head()` muncul |
| 2 | **Task 1** OLS pakai rumus (normal equation), TANPA sklearn/polyfit | β & α kecetak |
| 3 | **Task 2** OLS pakai gradient descent, tulis sendiri | β-nya ~sama Task 1 |
| 4 | Lapor β | β kira-kira **0.6** (0.55–0.70 masih wajar) |
| 5 | **Task 3** plot | ada scatter + garismu + garis y=x, garismu lebih landai |
| 6 | **Task 4** jawaban kata-kata | jawab pertanyaan prediksi 4 inci |
| 7 | **Task 5** (bonus) R² dari nol | angka R² kecetak |

Kalau semua kelar → `File → Save a copy in GitHub` ke repo ini → bilang ke gua **"ds-01 siap dicek"**. 
Gua jalanin notebook lu, cocokin ke rubric, terus tanya-tanya buat mastiin lu paham (bukan cuma nebak angka).


### ⚠️ Aturan main
- **Lu yang ngetik kodenya.** Gua udah jelasin rumus + langkahnya sedetail mungkin, tapi baris kode terakhir lu yang rakit.
- Boleh Google & baca dokumentasi numpy. **Nggak boleh** nyalin solusi orang bulat-bulat.
- Mentok? Ketik di sel apa yang lu bingung ("gua ga ngerti kenapa X.mean() ..."), nanti gua arahin — gua nggak bakal langsung kasih jawaban jadi.
- Kalau lu minta gua "tulisin aja kodenya", itu **nggak dihitung lulus**. 🙂


---
## 🔧 Setup — jalanin apa adanya (loading data BUKAN bagian latihan)

Klik sel di bawah, tekan **Shift+Enter** buat ngejalanin. Ini narik dataset asli Galton dari internet.
Apa yang terjadi baris per baris:
1. `import ...` — manggil alat: `numpy`=hitung angka, `pandas`=tabel, `matplotlib`=gambar grafik, `statsmodels`=sumber datanya.
2. `get_rdataset(...)` — download tabel `GaltonFamilies`.
3. Kita ambil 2 kolom aja: `midparentHeight` (rata-rata tinggi ortu) dan `childHeight` (tinggi anak).
4. `.to_numpy()` — ubah kolom tabel jadi **array numpy** (deretan angka) biar gampang dihitung. Namain `X` (input) & `y` (yang mau diprediksi).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Dataset asli Galton (via Rdatasets)
gf = sm.datasets.get_rdataset('GaltonFamilies', 'HistData').data
df = gf[['midparentHeight', 'childHeight']].dropna().reset_index(drop=True)
X = df['midparentHeight'].to_numpy()   # tinggi rata-rata ortu (inci)  -> INPUT
y = df['childHeight'].to_numpy()       # tinggi anak (inci)            -> TARGET
print('jumlah baris (n) =', len(df))
print('rata-rata tinggi ortu  =', round(X.mean(), 2), 'inci')
print('rata-rata tinggi anak  =', round(y.mean(), 2), 'inci')
df.head()

def cek(nama, got, lo, hi, kalau_salah=''):
    """Self-check: cetak OK kalau `got` masuk rentang target. Bukan penilai, cuma rambu."""
    ok = lo <= got <= hi
    print(f"{'OK ' if ok else 'X  '} {nama} = {got:.4f}   (target {lo}-{hi})")
    if not ok and kalau_salah:
        print(f'    -> {kalau_salah}')


---
## Task 1 — Cari garis terbaik pakai RUMUS (normal equation)

### 🧠 Konsep
"Garis terbaik" = garis yang **totalnya paling deket** ke semua titik. Ukuran "deket" yang dipakai: 
jumlahkan **(jarak vertikal tiap titik ke garis)²**, lalu cari α, β yang bikin jumlah itu **paling kecil**. 
Ini namanya *Ordinary Least Squares* (OLS) = "kuadrat-terkecil biasa". Kenapa dikuadratin? biar jarak minus/plus nggak saling nutup & yang meleset jauh kena penalti gede.

### 📐 Rumus (ini hasil turunan kalkulus — buat sekarang pakai aja)
Hitung **β dulu**, baru **α**:

```
        Σ (xᵢ − x̄)(yᵢ − ȳ)          <- 'atas': gimana X & y gerak bareng
  β  =  ─────────────────────
           Σ (xᵢ − x̄)²             <- 'bawah': seberapa nyebar si X

  α  =  ȳ − β · x̄
```
Baca simbol: **x̄** = rata-rata semua X (`X.mean()`), **ȳ** = rata-rata semua y (`y.mean()`), 
**Σ** = jumlahkan semua (`np.sum(...)`), **xᵢ** = tiap elemen X.

### 🪜 Pecah jadi langkah super kecil
1. `xbar = X.mean()` dan `ybar = y.mean()` → dua angka rata-rata.
2. `dx = X - xbar` → array selisih tiap X dari rata-ratanya. (numpy otomatis kurangin tiap elemen — nggak perlu loop!)
3. `dy = y - ybar` → sama buat y.
4. **Atas** = `np.sum(dx * dy)`  (`dx * dy` ngaliin elemen-per-elemen dulu, baru dijumlah).
5. **Bawah** = `np.sum(dx ** 2)`  (`** 2` = dikuadratin tiap elemen).
6. `beta = atas / bawah`.
7. `alpha = ybar - beta * xbar`.
8. `print` hasilnya.

### 🚫 Jangan pakai `sklearn` / `np.polyfit` di sini (itu "tombol jadi"). Boleh dipakai NANTI cuma buat ngecek.

Isi TODO di sel bawah (breadcrumb-nya udah gua nomorin sesuai langkah di atas):


In [ ]:
# Task 1 — OLS via normal equation (isi tiap TODO)
# 1) rata-rata
# xbar = ...
# ybar = ...

# 2-3) selisih dari rata-rata
# dx = ...
# dy = ...

# 4-5) atas & bawah pecahan
# atas  = ...
# bawah = ...

# 6-7) beta lalu alpha
# beta  = ...
# alpha = ...

# 8) tampilkan
# print(f'beta (kemiringan) = {beta:.4f}')
# print(f'alpha (intercept) = {alpha:.4f}')

# 9) self-check
# cek('beta', beta, 0.55, 0.70, 'X & y ketuker? atas/bawah pecahan kebalik?')


**🔎 Cek Task 1:** `beta` harusnya sekitar **0.6**. Kalau mau yakin, jalanin `np.polyfit(X, y, 1)` di sel terpisah — angka pertamanya (slope) harus sama kayak `beta` lu. (polyfit cuma buat ngecek, bukan jawaban.)


---
## Task 2 — Cari garis yang SAMA, tapi pakai Gradient Descent

### 🧠 Konsep: kenapa repot kalau rumus Task 1 udah dapet?
Karena rumus tertutup kayak Task 1 cuma ADA buat model gampang. Model gede (neural net, dll.) **nggak punya rumus jadi** — 
harus dicari pelan-pelan lewat *gradient descent* (GD). Jadi ini latihan buat senjata yang dipake seumur hidup.

Analogi GD: lu ditutup mata di lembah, mau ke titik terendah. Caranya: **raba kemiringan tanah di kaki lu, 
melangkah kecil ke arah menurun, ulang** sampai datar. "Kemiringan" itu = **gradient**. "Langkah kecil" itu = **learning rate**.

### 📐 Rumus yang lu butuh
Prediksi:  `ŷ = alpha + beta * X`  (topi = tebakan).  
Error tiap titik:  `e = ŷ - y`.  
"Ketinggian lembah" (loss, MSE) = `mean(e²)` — makin kecil makin bagus.

Kemiringan lembah ke arah tiap parameter (turunan MSE):
```
  grad_alpha = 2 * mean(e)          # e = (ŷ - y)
  grad_beta  = 2 * mean(e * X)
```
Aturan melangkah (ulang banyak kali):
```
  alpha = alpha - lr * grad_alpha
  beta  = beta  - lr * grad_beta
```

### ⚠️ 1 jebakan penting: skala X gede (~68). Itu bikin GD meledak (angka jadi `nan`).
Obatnya: **standarisasi** X dulu → `Xs = (X - X.mean()) / X.std()`. Latih di `Xs`, dapet `beta_s, alpha_s`, 
terus di akhir **balikin ke skala asli**:
```
  beta  = beta_s / X.std()
  alpha = alpha_s - beta * X.mean()
```
(Kalau lu males mikir konversi ini, boleh juga pakai `lr` super kecil ~0.0001 tanpa standarisasi — tapi lebih lambat & gampang meledak. Saran gua: standarisasi.)

### 🪜 Langkah kecil
1. `Xs = (X - X.mean()) / X.std()`.
2. `alpha_s, beta_s = 0.0, 0.0` (mulai dari nol).
3. `lr = 0.1`, `losses = []`.
4. Loop `for i in range(2000):`
   - a. `yhat = alpha_s + beta_s * Xs`
   - b. `e = yhat - y`
   - c. `alpha_s -= lr * 2 * e.mean()`
   - d. `beta_s  -= lr * 2 * (e * Xs).mean()`
   - e. `losses.append((e**2).mean())`
5. Balikin skala → `beta`, `alpha` (rumus di atas).
6. `print(beta, alpha)` dan plot `losses` (harus turun ke datar).

Isi TODO:


In [ ]:
# Task 2 — Gradient Descent (isi tiap TODO)
# 1) standarisasi X
# Xs = ...

# 2-3) inisialisasi
# alpha_s, beta_s = 0.0, 0.0
# lr = 0.1
# losses = []

# 4) loop update
# for i in range(2000):
#     yhat = ...
#     e    = ...
#     alpha_s -= ...
#     beta_s  -= ...
#     losses.append(...)

# 5) balikin ke skala asli
# beta  = ...
# alpha = ...

# 6) tampilkan + cek konvergensi
# print(f'GD  beta={beta:.4f}  alpha={alpha:.4f}')
# plt.plot(losses); plt.xlabel('iterasi'); plt.ylabel('loss (MSE)'); plt.title('harus turun'); plt.show()

# 7) self-check
# cek('beta (GD)', beta, 0.55, 0.70, 'nan/meledak? lr kegedean atau lupa standarisasi X')


**🔎 Cek Task 2:** `beta` dari GD harus **mepet banget** sama `beta` dari Task 1 (beda < 0.01). Kurva `losses` harus **turun lalu datar**. Kalau `beta` jadi `nan` atau meledak → `lr` kegedean, atau lupa standarisasi X.


---
## Task 3 — Gambar buktinya (plot)

### 🧠 Yang mau ditunjukin
Satu grafik berisi **3 hal**: (a) titik data asli, (b) garis regresi lu, (c) garis `y = x` (kalau anak = copy ortu). 
Kalau efek Galton bener, **garis regresi lu lebih landai** dari garis `y=x` — dua garis itu motong di sekitar rata-rata, terus garismu "turun" di kanan & "naik" di kiri relatif ke y=x.

### 📐 Cara gambar garis lurus di matplotlib
Garis butuh 2+ titik x lalu hitung y-nya:
```
  gx = np.array([X.min(), X.max()])     # ujung kiri & kanan
  gy = alpha + beta * gx                # y garis regresi di dua ujung itu
```

### 🪜 Langkah kecil
1. `plt.scatter(X, y, s=8, alpha=0.4)` → titik data (s=ukuran, alpha=transparansi biar numpuknya keliatan).
2. `plt.plot(gx, gy, 'r-', label='regresi (β≈0.6)')` → garis regresi merah.
3. `plt.plot(gx, gx, 'k--', label='y = x (β=1)')` → garis putus-putus hitam (y=x).
4. `plt.xlabel(...)`, `plt.ylabel(...)`, `plt.legend()`, `plt.show()`.

Isi TODO:


In [ ]:
# Task 3 — plot (isi tiap TODO)
# gx = ...
# gy = ...

# plt.scatter(...)
# plt.plot(gx, gy, 'r-', label=...)
# plt.plot(gx, gx, 'k--', label=...)
# plt.xlabel('tinggi rata-rata ortu (inci)')
# plt.ylabel('tinggi anak (inci)')
# plt.legend(); plt.show()


**🔎 Cek Task 3:** garis merah (regresi) harus **lebih landai** dari garis putus-putus (y=x). Itu tampilan visual dari "regression to the mean".


---
## Task 4 — Jawab pakai kata-kata (dobel-klik sel ini buat ngetik)

Pakai `beta` yang lu dapet. Prediksi selisih anak dari rata-rata = `beta × selisih ortu dari rata-rata`.

**Q1: Ortu 4 inci di atas rata-rata. Anak diprediksi berapa inci di atas rata-rata?** (hitung: `beta × 4` = ...)  
**Q2: Kenapa jawabannya lebih kecil dari 4 inci?** (hubungin ke β < 1 dan istilah "regression to the mean".)

> **Jawaban lu:**
> - Q1: ...
> - Q2: ...


---
## Task 5 — (Bonus) R²: seberapa bagus garisnya, dari nol

### 🧠 Konsep
**R²** = porsi variasi tinggi anak yang berhasil **dijelasin** sama garis lu. 0 = garis nggak ngasih info, 1 = sempurna. 
Buat Galton, wajar **rendah** (~0.2) — tinggi ortu cuma sebagian kecil cerita; sisanya faktor lain. Rendah itu BUKAN error, itu temuannya.

### 📐 Rumus
```
  SS_res = Σ (yᵢ − ŷᵢ)²      # sisa error setelah pakai garis   (ŷ = alpha + beta*X)
  SS_tot = Σ (yᵢ − ȳ)²       # error kalau nebak pakai rata-rata doang
  R² = 1 − SS_res / SS_tot
```

### 🪜 Langkah
1. `yhat = alpha + beta * X`.
2. `ss_res = np.sum((y - yhat) ** 2)`.
3. `ss_tot = np.sum((y - y.mean()) ** 2)`.
4. `r2 = 1 - ss_res / ss_tot`; `print`.

Isi TODO:


In [ ]:
# Task 5 — R² dari nol (isi tiap TODO)
# yhat   = ...
# ss_res = ...
# ss_tot = ...
# r2     = ...
# print(f'R^2 = {r2:.4f}')

# cek('R^2', r2, 0.05, 0.35, 'ss_res & ss_tot ketuker? yhat pakai alpha+beta*X?')


---
### 🔎 Self-check terakhir (sebelum bilang "siap dicek")
- [ ] `beta` ≈ **0.6** (0.55–0.70) di Task 1 **dan** Task 2, dan keduanya cocok.
- [ ] Kurva loss Task 2 turun ke datar (nggak `nan`).
- [ ] Plot: garis regresi lebih landai dari y=x.
- [ ] Task 4 kejawab dua-duanya.
- [ ] (bonus) R² kecetak.

Meleset jauh? cek dulu: X & y ketuker? pembilang/penyebut Task 1 kebalik? lupa standarisasi di Task 2? 
Masih bingung → tulis pertanyaannya di sel, `Save a copy in GitHub`, bilang **"ds-01 siap dicek"**. Gua review.
